In [ ]:
import sys
sys.path.insert(0, '..')
from src.portfolio import kalshi_order_price
from src.config import DB_PATH
import pandas as pd
import sqlite3
import re
from src.data import (
    load_and_prepare, get_username_mappings,
    load_standings, load_model_predictions,
)
from src.simulation import (
    build_player_pool, build_player_pool_score,
    run_simulation, run_simulation_score, build_results,
)
from src.portfolio import (
    build_portfolio, run_portfolio_mc, run_portfolio_mc_score, pnl_summary,
    run_payoff_matrix, portfolio_kelly,
    compute_hedge_correlations, show_best_hedges, kalshi_order_price,
)
USERNAME_TO_PLAYER, PLAYER_TO_USERNAME = get_username_mappings()

# Load all settled top-N and winner markets that have backtest predictions
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query("""
        SELECT
            kms.tournament_date,
            kms.event_ticker,
            kms.player,
            kms.yes_bid_close,
            kms.yes_ask_close,
            kms.open_interest,
            kms.settlement_value,
            bp.p_participate,
            bp.P_top1_given_play,
            bp.P_top3_given_play,
            bp.P_top5_given_play,
            bp.P_top8_given_play,
            bp.P_top10_given_play
        FROM kalshi_market_snapshots kms
        INNER JOIN backtest_predictions bp
            ON kms.tournament_date = bp.tourn_date
           AND kms.player = bp.player_name
        WHERE kms.open_interest > 0
          AND kms.yes_ask_close IS NOT NULL
          AND kms.yes_bid_close IS NOT NULL
          AND kms.settlement_value IS NOT NULL
    """, conn)

print(f'{len(df)} matched markets across {df["tournament_date"].nunique()} tournaments')
df.head()

152 matched markets across 9 tournaments


,tournament_date,event_ticker,player,yes_bid_close,yes_ask_close,open_interest,settlement_value,p_participate,P_top1_given_play,P_top3_given_play,P_top5_given_play,P_top8_given_play,P_top10_given_play
0,2026-05-26,KXTITLEDTUESDAY-26MAY26,Magnus Carlsen,0.02,0.10,404.96,0.0,0.321354,0.262968,0.449855,0.528083,0.621651,0.669571
1,2026-05-26,KXTITLEDTUESTOP-26MAY26T10,Aleksei Sarana,0.11,0.25,5.00,1.0,0.771070,0.060754,0.180783,0.282832,0.403588,0.456613
2,2026-05-26,KXTITLEDTUESTOP-26MAY26T10,Denis Lazavik,0.11,0.24,5.00,0.0,0.721659,0.068936,0.226496,0.338218,0.459924,0.531217
3,2026-05-26,KXTITLEDTUESTOP-26MAY26T10,Fabiano Caruana,0.19,0.59,15.00,1.0,0.419421,0.032331,0.121620,0.209879,0.321998,0.391423
4,2026-05-26,KXTITLEDTUESTOP-26MAY26T10,Hikaru Nakamura,0.31,0.75,320.00,0.0,0.652821,0.154974,0.344904,0.471020,0.624419,0.700202


In [43]:
# Parse N from ticker; winner markets (KXTITLEDTUESDAY-*) map to top-1
P_COL = {1: 'P_top1_given_play', 3: 'P_top3_given_play',
         5: 'P_top5_given_play', 8: 'P_top8_given_play',
         10: 'P_top10_given_play'}

def parse_n(ticker):
    m = re.search(r'T(\d+)$', ticker or '')
    if m:
        return int(m.group(1))
    return 1 if 'KXTITLEDTUESDAY-' in ticker else None

df['n'] = df['event_ticker'].apply(parse_n)
df['model_prob'] = pd.NA
for n, col in P_COL.items():
    mask = df['n'] == n
    df.loc[mask, 'model_prob'] = df.loc[mask, 'p_participate'] * df.loc[mask, col]

df['yes_ask_eff'] = df['yes_ask_close'].apply(kalshi_order_price)
df['no_ask_eff']  = (1.0 - df['yes_bid_close']).apply(kalshi_order_price)

df = df.dropna(subset=['model_prob'])
df[['tournament_date', 'player', 'n', 'model_prob', 'yes_ask_eff', 'no_ask_eff', 'settlement_value']].head()

,tournament_date,player,n,model_prob,yes_ask_eff,no_ask_eff,settlement_value
0,2026-05-26,Magnus Carlsen,1,0.084506,0.106300,0.981372,0.0
1,2026-05-26,Aleksei Sarana,10,0.352081,0.263125,0.896853,1.0
2,2026-05-26,Denis Lazavik,10,0.383357,0.252768,0.896853,0.0
3,2026-05-26,Fabiano Caruana,10,0.164171,0.606933,0.820773,1.0
4,2026-05-26,Hikaru Nakamura,10,0.457106,0.763125,0.704973,0.0


In [54]:
THRESHOLD = 1.25  # minimum model/price ratio to place a bet

df_yes = df[df['model_prob'] > df['yes_ask_eff'] * THRESHOLD].copy()
df_no  = df[df['yes_bid_close'] > df['model_prob'] * THRESHOLD].copy()

print(f'{len(df_yes)} YES positions, {len(df_no)} NO positions  '
      f'(threshold {THRESHOLD:.2f}x, {df["tournament_date"].nunique()} tournaments)')

17 YES positions, 28 NO positions  (threshold 1.25x, 9 tournaments)


In [55]:
df_yes['cost']   = df_yes['yes_ask_eff'] * df_yes['open_interest']
df_yes['return'] = df_yes['settlement_value'] * df_yes['open_interest']
df_no['cost']    = df_no['no_ask_eff'] * df_no['open_interest']
df_no['return']  = (1.0 - df_no['settlement_value']) * df_no['open_interest']

yes_cost, yes_ret = df_yes['cost'].sum(), df_yes['return'].sum()
no_cost,  no_ret  = df_no['cost'].sum(),  df_no['return'].sum()
total_cost   = yes_cost + no_cost
total_return = yes_ret + no_ret

print(f'{"":20} {"Bets":>6}  {"Cost ($)":>10}  {"Return ($)":>10}  {"P&L ($)":>10}')
print('-' * 62)
print(f'{"YES positions":<20} {len(df_yes):>6}  {yes_cost:>10.2f}  {yes_ret:>10.2f}  {yes_ret-yes_cost:>+10.2f}')
print(f'{"NO positions":<20} {len(df_no):>6}  {no_cost:>10.2f}  {no_ret:>10.2f}  {no_ret-no_cost:>+10.2f}')
print('-' * 62)
print(f'{"Total":<20} {len(df_yes)+len(df_no):>6}  {total_cost:>10.2f}  {total_return:>10.2f}  {total_return-total_cost:>+10.2f}')

                       Bets    Cost ($)  Return ($)     P&L ($)
--------------------------------------------------------------
YES positions            17      476.56      784.84     +308.28
NO positions             28     5409.71     5820.34     +410.63
--------------------------------------------------------------
Total                    45     5886.27     6605.18     +718.91


In [24]:
# Load all settled top-N and winner markets that have backtest predictions
with sqlite3.connect(DB_PATH) as conn:
    dfs = pd.read_sql_query("""
        SELECT * FROM (SELECT date, username, rank FROM titled_tuesday_standings
        WHERE date > '2026-07-27') s JOIN (SELECT tourn_date, username, p_participate, P_top1_given_play, P_top3_given_play, P_top5_given_play, P_top8_given_play FROM historical_predictions
    WHERE tourn_date > '2026-07-27') p ON s.username=p.username;
    """, conn)

dfs.head()

,date,username,rank,tourn_date,username,p_participate,P_top1_given_play,P_top3_given_play,P_top5_given_play,P_top8_given_play
0,2026-07-28 00:00:00,1assen,205.0,2026-07-28,1assen,0.339588,0.0,0.0,0.0,0.0
1,2026-07-28 00:00:00,APS,219.0,2026-07-28,APS,0.555564,0.0,0.0,0.0,0.0
2,2026-07-28 00:00:00,Aamuktha-Guntaka,243.0,2026-07-28,Aamuktha-Guntaka,0.043608,0.0,0.0,0.0,0.0
3,2026-07-28 00:00:00,Achuvettan,46.0,2026-07-28,Achuvettan,0.556295,0.0,0.0,0.0,0.0
4,2026-07-28 00:00:00,Actuary44,360.0,2026-07-28,Actuary44,0.528591,0.0,0.0,0.0,0.0


In [ ]:
pd.set_option('display.max_columns',None)
for k in [1, 3, 5, 8]:
    dfs[f'p_top{k}']=dfs['p_participate']*dfs[f'P_top{k}_given_play']
    dfs = dfs.drop(columns=f'P_top{k}_given_play')

dfs.sort_values('rank').head(40)

,date,username,rank,tourn_date,username,p_participate,p_top1,p_top3,p_top5,p_top8
63,2026-07-28 00:00:00,Firouzja2003,1.0,2026-07-28,Firouzja2003,0.129878,0.012132,0.028736,0.037842,0.046028
67,2026-07-28 00:00:00,GHANDEEVAM2003,2.0,2026-07-28,GHANDEEVAM2003,0.574030,0.034231,0.134318,0.218051,0.298225
323,2026-07-28 00:00:00,vi_pranav,3.0,2026-07-28,vi_pranav,0.591700,0.008851,0.023499,0.040250,0.082862
53,2026-07-28 00:00:00,Dr_Tyger,4.0,2026-07-28,Dr_Tyger,0.343603,0.003716,0.009005,0.020133,0.049689
139,2026-07-28 00:00:00,Msb2,5.0,2026-07-28,Msb2,0.783156,0.011117,0.049578,0.094698,0.152204
100,2026-07-28 00:00:00,Jospem,6.0,2026-07-28,Jospem,0.694236,0.038181,0.081074,0.121315,0.182489
318,2026-07-28 00:00:00,tptagain,7.0,2026-07-28,tptagain,0.353927,0.001849,0.007605,0.015160,0.031980
34,2026-07-28 00:00:00,BogdanDeac,8.0,2026-07-28,BogdanDeac,0.463326,0.005840,0.023021,0.054952,0.115714
51,2026-07-28 00:00:00,DenLaz,9.0,2026-07-28,DenLaz,0.863765,0.074715,0.214874,0.307230,0.393635
82,2026-07-28 00:00:00,Grischuk,10.0,2026-07-28,Grischuk,0.703387,0.008565,0.045618,0.097646,0.166457
